In [1]:
# !pip install -q torch-scatter -f https://data.pyg.org/whl/torch-1.10.0+cu113.html
# !pip install -q torch-sparse -f https://data.pyg.org/whl/torch-1.10.0+cu113.html
# !pip install -q torch-cluster -f https://data.pyg.org/whl/torch-1.10.0+cu113.html
# !pip install -q git+https://github.com/pyg-team/pytorch_geometric.git

In [22]:
import numpy as np
import torch

In [23]:
from torch_geometric.data import HeteroData

In [38]:
authors = torch.rand((10,8))
papers = torch.rand((20,4))

authors_y = torch.rand(10).round()

write_from = torch.tensor(np.random.choice(10,50,replace=True))
write_to = torch.tensor(np.random.choice(20,50,replace=True))
write = torch.concat((write_from,write_to)).reshape(-1,50).long()

cite_from = torch.tensor(np.random.choice(20,15,replace=True))
cite_to = torch.tensor(np.random.choice(20,15,replace=True))
cite = torch.concat((cite_from,cite_to)).reshape(-1,15).long()

write_reversed = torch.stack((write[1], write[0]), dim=0)


In [39]:
#data = HeteroData ({'author':{'x':authors, 'y':authors_y}, 'paper':{'x':papers}}, author_write_paper={'edge_index': write}, paper_cite_paper={'edge_index' : cite})

# data = HeteroData({'author': {'x': authors, 'y': authors_y}, 'paper': {'x': papers},
#     ('author', 'write', 'paper'): {'edge_index': write},
#     ('paper', 'cite', 'paper'): {'edge_index': cite}
# })

data = HeteroData({
    'author': {'x': authors, 'y': authors_y},
    'paper': {'x': papers},
    ('author', 'write', 'paper'): {'edge_index': write}, # Renamed for clarity
    ('paper', 'cite', 'paper'): {'edge_index': cite},
    ('paper', 'is_written_by', 'author'): {'edge_index': write_reversed}  # ADDED REVERSE EDGE
})

In [40]:
data.metadata()

(['author', 'paper'],
 [('author', 'writes', 'paper'),
  ('paper', 'cites', 'paper'),
  ('paper', 'is_written_by', 'author')])

In [41]:
data['author']

{'x': tensor([[0.4511, 0.0327, 0.1291, 0.8159, 0.4731, 0.1815, 0.5525, 0.5090],
        [0.5487, 0.6382, 0.5940, 0.3516, 0.0180, 0.6936, 0.8408, 0.8576],
        [0.1689, 0.2462, 0.7231, 0.2703, 0.3637, 0.8914, 0.6215, 0.2610],
        [0.2309, 0.0517, 0.4854, 0.8788, 0.8412, 0.1124, 0.0679, 0.7331],
        [0.2647, 0.1588, 0.6357, 0.7979, 0.9940, 0.7078, 0.8959, 0.0878],
        [0.4243, 0.8221, 0.9329, 0.6071, 0.9846, 0.8346, 0.8560, 0.8356],
        [0.8976, 0.8139, 0.7302, 0.6710, 0.1391, 0.6102, 0.1143, 0.2480],
        [0.6155, 0.6249, 0.0239, 0.0140, 0.3493, 0.4487, 0.5268, 0.6431],
        [0.0814, 0.3291, 0.9319, 0.8709, 0.4799, 0.1100, 0.2802, 0.6441],
        [0.7442, 0.8228, 0.1498, 0.7922, 0.5259, 0.6741, 0.5765, 0.5306]]), 'y': tensor([1., 0., 1., 1., 1., 1., 1., 0., 0., 1.])}

In [42]:
data.x_dict

{'author': tensor([[0.4511, 0.0327, 0.1291, 0.8159, 0.4731, 0.1815, 0.5525, 0.5090],
         [0.5487, 0.6382, 0.5940, 0.3516, 0.0180, 0.6936, 0.8408, 0.8576],
         [0.1689, 0.2462, 0.7231, 0.2703, 0.3637, 0.8914, 0.6215, 0.2610],
         [0.2309, 0.0517, 0.4854, 0.8788, 0.8412, 0.1124, 0.0679, 0.7331],
         [0.2647, 0.1588, 0.6357, 0.7979, 0.9940, 0.7078, 0.8959, 0.0878],
         [0.4243, 0.8221, 0.9329, 0.6071, 0.9846, 0.8346, 0.8560, 0.8356],
         [0.8976, 0.8139, 0.7302, 0.6710, 0.1391, 0.6102, 0.1143, 0.2480],
         [0.6155, 0.6249, 0.0239, 0.0140, 0.3493, 0.4487, 0.5268, 0.6431],
         [0.0814, 0.3291, 0.9319, 0.8709, 0.4799, 0.1100, 0.2802, 0.6441],
         [0.7442, 0.8228, 0.1498, 0.7922, 0.5259, 0.6741, 0.5765, 0.5306]]),
 'paper': tensor([[0.6863, 0.3690, 0.9879, 0.9598],
         [0.1806, 0.1389, 0.9336, 0.8573],
         [0.0549, 0.6598, 0.9970, 0.1932],
         [0.8514, 0.1091, 0.6250, 0.5340],
         [0.0334, 0.9549, 0.2975, 0.7395],
         [0.07

In [43]:
homogeneous_data = data.to_homogeneous()

In [44]:
homogeneous_data

Data(edge_index=[2, 115], x=[30, 8], y=[30], node_type=[30], edge_type=[115])

In [45]:
data.node_stores

[{'x': tensor([[0.4511, 0.0327, 0.1291, 0.8159, 0.4731, 0.1815, 0.5525, 0.5090],
         [0.5487, 0.6382, 0.5940, 0.3516, 0.0180, 0.6936, 0.8408, 0.8576],
         [0.1689, 0.2462, 0.7231, 0.2703, 0.3637, 0.8914, 0.6215, 0.2610],
         [0.2309, 0.0517, 0.4854, 0.8788, 0.8412, 0.1124, 0.0679, 0.7331],
         [0.2647, 0.1588, 0.6357, 0.7979, 0.9940, 0.7078, 0.8959, 0.0878],
         [0.4243, 0.8221, 0.9329, 0.6071, 0.9846, 0.8346, 0.8560, 0.8356],
         [0.8976, 0.8139, 0.7302, 0.6710, 0.1391, 0.6102, 0.1143, 0.2480],
         [0.6155, 0.6249, 0.0239, 0.0140, 0.3493, 0.4487, 0.5268, 0.6431],
         [0.0814, 0.3291, 0.9319, 0.8709, 0.4799, 0.1100, 0.2802, 0.6441],
         [0.7442, 0.8228, 0.1498, 0.7922, 0.5259, 0.6741, 0.5765, 0.5306]]), 'y': tensor([1., 0., 1., 1., 1., 1., 1., 0., 0., 1.])},
 {'x': tensor([[0.6863, 0.3690, 0.9879, 0.9598],
         [0.1806, 0.1389, 0.9336, 0.8573],
         [0.0549, 0.6598, 0.9970, 0.1932],
         [0.8514, 0.1091, 0.6250, 0.5340],
        

In [46]:
data.to_dict()

{'_global_store': {},
 'author': {'x': tensor([[0.4511, 0.0327, 0.1291, 0.8159, 0.4731, 0.1815, 0.5525, 0.5090],
          [0.5487, 0.6382, 0.5940, 0.3516, 0.0180, 0.6936, 0.8408, 0.8576],
          [0.1689, 0.2462, 0.7231, 0.2703, 0.3637, 0.8914, 0.6215, 0.2610],
          [0.2309, 0.0517, 0.4854, 0.8788, 0.8412, 0.1124, 0.0679, 0.7331],
          [0.2647, 0.1588, 0.6357, 0.7979, 0.9940, 0.7078, 0.8959, 0.0878],
          [0.4243, 0.8221, 0.9329, 0.6071, 0.9846, 0.8346, 0.8560, 0.8356],
          [0.8976, 0.8139, 0.7302, 0.6710, 0.1391, 0.6102, 0.1143, 0.2480],
          [0.6155, 0.6249, 0.0239, 0.0140, 0.3493, 0.4487, 0.5268, 0.6431],
          [0.0814, 0.3291, 0.9319, 0.8709, 0.4799, 0.1100, 0.2802, 0.6441],
          [0.7442, 0.8228, 0.1498, 0.7922, 0.5259, 0.6741, 0.5765, 0.5306]]),
  'y': tensor([1., 0., 1., 1., 1., 1., 1., 0., 0., 1.])},
 'paper': {'x': tensor([[0.6863, 0.3690, 0.9879, 0.9598],
          [0.1806, 0.1389, 0.9336, 0.8573],
          [0.0549, 0.6598, 0.9970, 0.1932

In [47]:
import torch_geometric.transforms as T
from torch_geometric.nn import Sequential, Linear
from torch.nn import ReLU

In [48]:
transform = T.RandomNodeSplit()
data = transform(data)

In [49]:
data

HeteroData(
  author={
    x=[10, 8],
    y=[10],
    train_mask=[10],
    val_mask=[10],
    test_mask=[10],
  },
  paper={ x=[20, 4] },
  (author, writes, paper)={ edge_index=[2, 50] },
  (paper, cites, paper)={ edge_index=[2, 15] },
  (paper, is_written_by, author)={ edge_index=[2, 50] }
)

In [50]:
import torch_geometric.transforms as T
from torch_geometric.datasets import OGB_MAG
from torch_geometric.nn import SAGEConv, to_hetero

class GNN(torch.nn.Module):
    def __init__(self, hidden_channels, out_channels):
        super().__init__()
        self.conv1 = SAGEConv((-1,-1), hidden_channels)
        self.conv2 = SAGEConv((-1,-1), out_channels)

    def forward(self, x, edge_index):
        x = torch.relu(self.conv1(x, edge_index))
        x = self.conv2(x, edge_index)
        return x

model = GNN(hidden_channels=64, out_channels=2)
model = to_hetero(model, data.metadata(), aggr='sum')